Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.


In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("Why do parrots talk")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots can talk. Let me start by recalling what I know about parrots. They\'re birds, right? And some species are known for mimicking human speech, like budgerigars or macaws. But why do they do that? Is it a natural behavior, or is it learned?\n\nFirst, maybe I should think about their anatomy. Do they have a special vocal system? I remember that birds have something called a syrinx, which is their vocal organ. Humans have vocal cords. Maybe the syrinx allows them to produce a wide range of sounds. So, maybe the structure of their syrinx is more flexible, enabling them to mimic sounds.\n\nAnother angle is their intelligence. Parrots are considered highly intelligent. They can solve problems and have social learning. Maybe talking is a way they communicate with humans, similar to how they might use other sounds in the wild. In the wild, parrots live in flocks, so communication is important for social bonding. If they\'re in

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the wather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])



In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")

print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': '924s7mwef', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 155, 'total_tokens': 241, 'completion_time': 0.13875599, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.019844547, 'prompt_tokens_details': None, 'queue_time': 0.051860136, 'total_time': 0.158600537}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e643e-bd8f-7282-aa

Tool Execution Loops

In [5]:
# Step 1: Model generates tool calls
messages =[{"role":"user","content":"What's the waether in Boston?"}]
ai_msg=model_with_tools.invoke(messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#Step 3: Pass results back to model for final response
final_response=model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72F and sunny."


It's sunny in Boston! ☀️ Let me know if you need more details.
